
# CH5 - Exercise 7

In Sections 5.1.2 and 5.1.3, we saw that the `cross_validate()` function
can be used in order to compute the LOOCV test error estimate.
Alternatively, one could compute those quantities using just `sm.GLM()`
and the `predict()` method of the fitted model within a for loop. You
will now take this approach in order to compute the LOOCV error
for a simple logistic regression model on the `Weekly` data set. Recall
that in the context of classification problems, the LOOCV error is
given in (5.4).

**(a)** Fit a logistic regression model that predicts `Direction` using `Lag1`
and `Lag2`.

**(b)** Fit a logistic regression model that predicts `Direction` using `Lag1`
and `Lag2` *using all but the first observation*.

**(c)** Use the model from (b) to predict the direction of the first observation.
You can do this by predicting that the first observation
will go up if $P(\texttt{Direction = "Up"} \mid \texttt{Lag1, Lag2}) > 0.5$. Was this
observation correctly classified?

**(d)** Write a for loop from $i = 1$ to $i = n$, where $n$ is the number of
observations in the data set, that performs each of the following steps:

- i. Fit a logistic regression model using all but the $i$th observation to predict `Direction` using `Lag1` and `Lag2`.
- ii. Compute the posterior probability of the market moving up for the $i$th observation.
- iii. Use the posterior probability for the $i$th observation in order to predict whether or not the market moves up.
- iv. Determine whether or not an error was made in predicting the direction for the $i$th observation. If an error was made, then indicate this as a `1`, and otherwise indicate it as a `0`.

**(e)** Take the average of the $n$ numbers obtained in (d)iv in order to
obtain the LOOCV estimate for the test error. Comment on the
results.

In [1]:
import numpy as np
import pandas as pd
from matplotlib .pyplot import subplots
import statsmodels .api as sm
from ISLP import load_data
from ISLP.models import ( ModelSpec as MS ,
summarize )

from ISLP import confusion_table
from ISLP.models import contrast
from sklearn. discriminant_analysis import \
( LinearDiscriminantAnalysis as LDA ,
QuadraticDiscriminantAnalysis as QDA)
from sklearn. naive_bayes import GaussianNB
from sklearn. neighbors import KNeighborsClassifier
from sklearn. preprocessing import StandardScaler
from sklearn. model_selection import train_test_split
from sklearn. linear_model import LogisticRegression

from functools import partial
from sklearn.model_selection import (
    train_test_split, cross_validate, KFold, ShuffleSplit
)
from sklearn import clone 
from ISLP.models import sklearn_sm


In [2]:
Weekly = load_data("Weekly")
Weekly.head()

,Year,Lag1,Lag2,Lag3,Lag4,Lag5,Volume,Today,Direction
0,1990,0.816,1.572,-3.936,-0.229,-3.484,0.154976,-0.270,Down
1,1990,-0.270,0.816,1.572,-3.936,-0.229,0.148574,-2.576,Down
2,1990,-2.576,-0.270,0.816,1.572,-3.936,0.159837,3.514,Up
3,1990,3.514,-2.576,-0.270,0.816,1.572,0.161630,0.712,Up
4,1990,0.712,3.514,-2.576,-0.270,0.816,0.153728,1.178,Up


### Model fitted over the whole dataset  

In [3]:
design  = MS(['Lag1', 'Lag2'])
X = design.fit_transform(Weekly)
y = (Weekly['Direction'] == 'Up')

model_tot   = sm.GLM(y, X, family=sm.families.Binomial())
results_tot = model_tot.fit()
summarize(results_tot)



,coef,std err,z,P>|z|
intercept,0.2212,0.061,3.599,0.000
Lag1,-0.0387,0.026,-1.477,0.140
Lag2,0.0602,0.027,2.270,0.023


### All the dataset but the first observation

In [4]:

X = design.fit_transform(Weekly[1:])
y = (Weekly['Direction'][1:] == 'Up')

model_1_missing   = sm.GLM(y, X, family=sm.families.Binomial())
results_1_missing = model_1_missing.fit()
summarize(results_1_missing)


,coef,std err,z,P>|z|
intercept,0.2232,0.061,3.630,0.000
Lag1,-0.0384,0.026,-1.466,0.143
Lag2,0.0608,0.027,2.291,0.022


Now we predict the fist observation

In [16]:
x_first = design.transform(Weekly.iloc[:1])
y_first = (Weekly['Direction'].iloc[0] == 'Up')

prob    = results_1_missing.predict(x_first)[0]
pred    = (prob > 0.5)

print(f'Predicted: {"Up" if pred else "Down"}')
print(f'True:      {"Up" if y_first else "Down"}')
print(f'Error:     {int(pred != y_first)}')

Predicted: Up
True:      Down
Error:     1


In [26]:
def logit_fn(Data, preds, idx):

    design = MS(preds)
    X      = design.fit_transform(Data.drop(index=Data.index[idx]))
    y      = (Data['Direction'].drop(index=Data.index[idx]) == 'Up')

    results = sm.GLM(y, X, family=sm.families.Binomial()).fit()

    x_i    = design.transform(Data.iloc[[idx]])
    y_i    = (Data['Direction'].iloc[idx] == 'Up')
    pred_i = results.predict(x_i).values[0]

    return int((pred_i > 0.5) != y_i)

#logit_fn(Data=Weekly, preds=['Lag1', 'Lag2'], idx=0)

errors = np.zeros(Weekly.shape[0])
for i in range(Weekly.shape[0]):
    errors[i] = logit_fn(Data=Weekly,
                            preds=['Lag1', 'Lag2'],
                            idx=i)

print('LOOCV error: ',np.mean(errors))





LOOCV error:  0.44995408631772266
